In [1]:
# W5D5 — Local Q&A Bot
# Cell 1 — Install requirements

!apt-get update -qq
!apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q requests chromadb sentence-transformers

print("Installation completed!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 M

In [2]:
# Cell 2 — Start Ollama server

import subprocess
import time
import shutil

ollama_path = shutil.which("ollama")

print("Ollama path:", ollama_path)

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started!")

Ollama path: /usr/local/bin/ollama
Ollama server started!


In [3]:
# Cell 3 — Verify Ollama

!ollama --version

ollama version is 0.34.0


In [4]:
# Cell 4 — Download Llama 3.2 3B

!ollama pull llama3.2:3b

print("llama3.2:3b is ready!")


llama3.2:3b is ready!


In [5]:
# Cell 5 — Check installed models

!ollama list

NAME           ID              SIZE      MODIFIED       
llama3.2:3b    a80c4f17acd5    2.0 GB    12 seconds ago    


In [6]:
# Cell 6 — First local inference

import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Explain machine learning in simple words.",
        "stream": False
    }
)

print("Llama 3.2 Response:\n")
print(response.json()["response"])

Llama 3.2 Response:

Machine learning is a way for computers to learn and get better at doing things on their own, without being explicitly programmed. It's like how you learn new things and get better at doing them too!

Here's how it works:

1. **Data is collected**: Computers are given lots of data, like pictures, words, or sounds.
2. **Patterns are found**: The computer looks at the data and tries to find patterns or relationships between things.
3. **Model is created**: The computer creates a model, like a recipe, that explains how the patterns work.
4. **Model is trained**: The computer uses the data to train the model, so it can make predictions or do things on its own.
5. **Model improves**: The computer keeps using the data to improve the model, so it can get better and better at doing things.

Machine learning can be used for things like:

* Image recognition (like recognizing a picture of a cat)
* Speech recognition (like recognizing a voice saying "hello")
* Predicting what

In [9]:
# Cell 7 — Ollama API function with custom system prompt

def ask_ollama(prompt):
    system_prompt = """
You are a helpful AI and Machine Learning mentor.

The user is a beginner.
Explain concepts in simple and clear language.
Give practical examples when useful.
Keep answers concise and easy to understand.
"""

    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "system": system_prompt,
            "prompt": prompt,
            "stream": False
        }
    )

    if response.status_code == 200:
        return response.json()["response"]
    else:
        return f"API Error: {response.text}"

In [10]:
# Cell 8 — Test 5 prompts

prompts = [
    "What is Python?",
    "What is machine learning?",
    "What is an embedding?",
    "What is a vector database?",
    "What is semantic search?"
]

for i, prompt in enumerate(prompts, 1):
    print("=" * 60)
    print(f"Prompt {i}: {prompt}")
    print("-" * 60)
    print(ask_ollama(prompt))
    print()

Prompt 1: What is Python?
------------------------------------------------------------
**What is Python?**

Python is a programming language that's easy to learn and fun to use. It's a high-level language, which means it abstracts away many low-level details, allowing you to focus on the logic of your program without getting bogged down in technical details.

**Example Use Case:**

Think of Python like a recipe book. You can use the book to write down a list of ingredients and instructions, and then use the steps to cook a delicious meal. In programming, Python is like the recipe book that helps you write down your ideas and instructions, and then the computer follows those steps to get the desired outcome.

**Key Features:**

* Easy to read and write
* Fast development and prototyping
* Extensive libraries and frameworks for various tasks (e.g., data analysis, web development, machine learning)
* Cross-platform compatible (works on Windows, macOS, Linux, etc.)

**Getting Started:**

Y

In [11]:
# Cell 9 — Download Qwen 2.5 3B

!ollama pull qwen2.5:3b

print("qwen2.5:3b is ready!")


qwen2.5:3b is ready!


In [12]:
# Cell 10 — Verify both models

!ollama list

NAME           ID              SIZE      MODIFIED       
qwen2.5:3b     357c53fb659c    1.9 GB    4 seconds ago     
llama3.2:3b    a80c4f17acd5    2.0 GB    12 minutes ago    


In [13]:
# Cell 11 — Compare models

def ask_model(model, prompt):

    system_prompt = """
You are a helpful AI mentor.
Answer clearly and simply for a beginner.
"""

    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": model,
            "system": system_prompt,
            "prompt": prompt,
            "stream": False
        }
    )

    if response.status_code == 200:
        return response.json()["response"]
    else:
        return f"Error: {response.text}"

In [14]:
# Cell 12 — Same 3 questions for both models

comparison_questions = [
    "Explain machine learning in simple words.",
    "What is the difference between AI and machine learning?",
    "Explain embeddings with a simple example."
]

print("Comparison questions ready!")

Comparison questions ready!


In [15]:
# Cell 13 — Compare responses

comparison_results = []

for i, question in enumerate(comparison_questions, 1):

    print("=" * 70)
    print(f"QUESTION {i}: {question}")
    print("=" * 70)

    llama_response = ask_model(
        "llama3.2:3b",
        question
    )

    qwen_response = ask_model(
        "qwen2.5:3b",
        question
    )

    print("\nLLAMA 3.2:\n")
    print(llama_response)

    print("\nQWEN 2.5:\n")
    print(qwen_response)

    comparison_results.append({
        "Question": question,
        "Llama 3.2": llama_response,
        "Qwen 2.5": qwen_response
    })

QUESTION 1: Explain machine learning in simple words.

LLAMA 3.2:

**What is Machine Learning?**

Machine learning is a way for computers to learn from data and make decisions on their own, without being told exactly what to do.

**Think of it like this:**

Imagine you're teaching a child to recognize different pictures. You show them many pictures of cats and dogs, and say "this is a cat" or "this is a dog". The child looks at the pictures, tries to understand the patterns, and eventually starts to recognize the difference between cats and dogs on their own.

In machine learning, the computer looks at many examples (like pictures of cats and dogs) and tries to understand the patterns too. Then, when it sees a new picture, it uses what it's learned to make a decision (like "this is a cat" or "this is a dog").

**Key benefits:**

1. **Automation**: Machines can do tasks faster and more accurately than humans.
2. **Improved accuracy**: Machines can learn from large amounts of data and im

In [19]:
# Cell 14 — Create Comparison DataFrame

import pandas as pd

comparison_df = pd.DataFrame(comparison_results)

print("Comparison DataFrame created successfully!")

display(comparison_df)

Comparison DataFrame created successfully!


,Question,Llama 3.2,Qwen 2.5
0,Explain machine learning in simple words.,**What is Machine Learning?**\n\nMachine learn...,Machine learning is like teaching a computer t...
1,What is the difference between AI and machine ...,AI (Artificial Intelligence) and Machine Learn...,AI is a broad term for making machines think a...
2,Explain embeddings with a simple example.,**What are Embeddings?**\n\nImagine you want t...,Sure! Let's think about embeddings in a simple...


In [20]:
# Cell 15 — Save model comparison

comparison_df.to_csv(
    "W5D5_Llama_vs_Qwen_Comparison.csv",
    index=False
)

print("Comparison saved successfully!")
print("File: W5D5_Llama_vs_Qwen_Comparison.csv")

Comparison saved successfully!
File: W5D5_Llama_vs_Qwen_Comparison.csv


In [21]:
# Cell 16 — Create ChromaDB collection

import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

client = chromadb.Client()

qa_collection = client.get_or_create_collection(
    name="w5d5_qa_documents",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB collection created!


In [22]:
# Cell 17 — Add knowledge base

knowledge_documents = [
    "Python is a high-level programming language widely used in software development, automation, data science and artificial intelligence.",

    "Machine learning is a branch of artificial intelligence where computers learn patterns from data to make predictions or decisions.",

    "Deep learning uses artificial neural networks with multiple layers to learn complex patterns.",

    "Natural language processing allows computers to process and understand human language.",

    "Embeddings are numerical vector representations of data such as text. Similar meanings generally produce similar vectors.",

    "A vector database stores vector representations and allows efficient similarity searches.",

    "ChromaDB is an open-source vector database commonly used for storing and retrieving embeddings in AI applications.",

    "Semantic search retrieves information based on meaning rather than relying only on exact keyword matching.",

    "Cosine similarity measures the similarity between two vectors based on the angle between them.",

    "Retrieval Augmented Generation, or RAG, retrieves relevant information and provides it as context to a language model."
]

knowledge_embeddings = embedding_model.encode(
    knowledge_documents
).tolist()

qa_collection.add(
    ids=[f"knowledge_{i}" for i in range(len(knowledge_documents))],
    documents=knowledge_documents,
    embeddings=knowledge_embeddings
)

print("Knowledge base added!")
print("Documents:", qa_collection.count())

Knowledge base added!
Documents: 10


In [23]:
# Cell 18 — Local Q&A Bot using ChromaDB + Ollama

def local_qa_bot(question):

    # Create question embedding
    question_embedding = embedding_model.encode(
        question
    ).tolist()

    # Retrieve relevant documents
    results = qa_collection.query(
        query_embeddings=[question_embedding],
        n_results=3
    )

    retrieved_documents = results["documents"][0]

    # Create context
    context = "\n\n".join(retrieved_documents)

    # Create prompt
    prompt = f"""
Use only the following context to answer the question.

Context:
{context}

Question:
{question}

Give a simple and accurate answer.
"""

    # Send to Ollama
    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False
        }
    )

    if response.status_code == 200:
        answer = response.json()["response"]

        return retrieved_documents, answer

    return retrieved_documents, "Ollama API error."

In [24]:
# Cell 19 — Test the Q&A bot

question = "What is semantic search?"

retrieved_documents, answer = local_qa_bot(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED DOCUMENTS:")
for i, document in enumerate(retrieved_documents, 1):
    print(f"\n{i}. {document}")

print("\n" + "=" * 60)

print("\nOLLAMA ANSWER:")
print(answer)

QUESTION:
What is semantic search?

RETRIEVED DOCUMENTS:

1. Semantic search retrieves information based on meaning rather than relying only on exact keyword matching.

2. Retrieval Augmented Generation, or RAG, retrieves relevant information and provides it as context to a language model.

3. Natural language processing allows computers to process and understand human language.


OLLAMA ANSWER:
Semantic search is a method of retrieving information based on meaning rather than exact keyword matching.


In [25]:
# Cell 20 — Test multiple Q&A questions

questions = [
    "What is machine learning?",
    "What are embeddings?",
    "What is ChromaDB?",
    "What is semantic search?",
    "What is RAG?"
]

for question in questions:

    _, answer = local_qa_bot(question)

    print("=" * 60)
    print("Question:", question)
    print("Answer:", answer)
    print()

Question: What is machine learning?
Answer: Machine learning is a branch of artificial intelligence where computers learn patterns from data to make predictions or decisions.

Question: What are embeddings?
Answer: Embeddings are numerical vector representations of data, such as text, that capture their meaning.

Question: What is ChromaDB?
Answer: ChromaDB is an open-source vector database.

Question: What is semantic search?
Answer: Semantic search is a method of retrieving information based on the meaning of words, rather than just exact keyword matching.

Question: What is RAG?
Answer: Based on the given context, there is no direct information about what RAG is. The context only mentions Retrieval Augmented Generation, but its full name is not provided, and its definition is not given.



In [26]:
# Cell 21 — Final verification

print("=" * 70)
print("W5D5 — LOCAL Q&A BOT FINAL VERIFICATION")
print("=" * 70)

print("\n✅ Ollama installed")
print("✅ llama3.2:3b tested")
print("✅ Custom system prompt created")
print("✅ Five prompts tested")
print("✅ qwen2.5:3b tested")
print("✅ Three-question model comparison completed")
print("✅ ChromaDB knowledge base created")
print("✅ Semantic retrieval completed")
print("✅ Retrieved context passed to Ollama")
print("✅ Local Q&A Bot tested")

print("\n" + "=" * 70)
print("🎉 W5D5 PROJECT COMPLETED!")
print("=" * 70)

W5D5 — LOCAL Q&A BOT FINAL VERIFICATION

✅ Ollama installed
✅ llama3.2:3b tested
✅ Custom system prompt created
✅ Five prompts tested
✅ qwen2.5:3b tested
✅ Three-question model comparison completed
✅ ChromaDB knowledge base created
✅ Semantic retrieval completed
✅ Retrieved context passed to Ollama
✅ Local Q&A Bot tested

🎉 W5D5 PROJECT COMPLETED!
